# PPMS 输运数据分析 Notebook

这个 Notebook 是一个**离线、只读输入**的分析入口。它读取 SQLite 或 MultiVu ETO `.dat`，调用 `ppms_control.plotting` 的公共 API，并把图写入指定输出目录。

它不会连接 SR830、SR865A、Keithley 或 PPMS，也不会执行真实硬件命令。真实扫描在 PowerShell 中运行；第二个 PowerShell 窗口使用 `monitor-run` 查看最近一次已提交状态。

推荐每次修改顶部参数后执行 **Kernel → Restart Kernel and Run All Cells**，避免使用旧 cell 状态。

## 0. 扫描—监视—分析工作流

1. 修改并保存 TOML。
2. PowerShell 窗口 1：配置校验、诊断、仿真或授权真实扫描。
3. PowerShell 窗口 2：`monitor-run ... --latest-running` 只读查看 SQLite。
4. 扫描完成后，在本 Notebook 中选择 SQLite `run_id` 或 ETO 路径。
5. 先检查质量警告，再生成、筛选和导出图。

下面的命令助手只打印命令供复制，不执行命令。按 `Ctrl+C` 停止监视器不会停止测量；只有在控制窗口按 `Ctrl+C` 才会中断测量并触发安全清理。

In [ ]:
from __future__ import annotations

from collections import Counter
import math
from pathlib import Path
import sys
import tomllib

from IPython.display import FileLink, Image, Markdown, display
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / 'pyproject.toml').is_file():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'pyproject.toml').is_file():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError('请从 ppms_qcodes_control 根目录或 notebooks 目录打开本 Notebook。')

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from ppms_control.plotting import (
    generate_publication_plots,
    list_sqlite_runs,
    load_eto_path,
    load_gate_calibration,
    load_sqlite_run,
)

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

## 1. PowerShell 命令助手（只打印，不执行）

修改 `SCAN_KIND` 后运行本 cell。可选：`voltage`、`frequency`、`field`、`temperature_field`、`gate`。其中 voltage 扫描的是 SR830 输出电压；电流仍是通过串联电阻得到的估算值。监视数据库从对应 TOML 的 `[data].database_path` 只读解析；真实运行前必须先创建并确认 `hardware.local.toml`。角度控制和 ETO sequence 控制尚未实现。

In [ ]:
SCAN_KIND = 'gate'
SIMULATION_CONFIG = PROJECT_ROOT / 'config' / 'simulation.toml'
HARDWARE_CONFIG = PROJECT_ROOT / 'config' / 'hardware.local.toml'
DIAGNOSTIC_RUN_ID = '<DIAGNOSTIC_RUN_ID>'

simulate_suffix = {
    'voltage': '',
    'frequency': '-frequency',
    'field': '-field',
    'temperature_field': '-temperature-field',
    'gate': '-gate',
}
hardware_suffix = simulate_suffix
if SCAN_KIND not in simulate_suffix:
    raise ValueError(f'Unsupported SCAN_KIND: {SCAN_KIND}')

def ps_quote(value: object) -> str:
    return "'" + str(value).replace("'", "''") + "'"

def database_from_config(config_path: Path) -> Path | None:
    if not config_path.is_file():
        return None
    with config_path.open('rb') as handle:
        raw = tomllib.load(handle)
    value = Path(raw['data']['database_path']).expanduser()
    return value.resolve() if value.is_absolute() else (config_path.parent / value).resolve()

python = f"& {ps_quote(sys.executable)} -m ppms_control"
print('1) Validate simulation config:')
print(f"{python} validate-config {ps_quote(SIMULATION_CONFIG)}")
print('\n2) Run simulation:')
print(f"{python} simulate{simulate_suffix[SCAN_KIND]} {ps_quote(SIMULATION_CONFIG)}")
print('\n3) Validate and diagnose real hardware:')
print(f"{python} validate-config {ps_quote(HARDWARE_CONFIG)}")
print(f"{python} diagnose-hardware {ps_quote(HARDWARE_CONFIG)}")
print('\n4) Authorized real scan (replace diagnostic ID):')
print(
    f"{python} run-hardware{hardware_suffix[SCAN_KIND]} {ps_quote(HARDWARE_CONFIG)} "
    f"--diagnostic-run-id {ps_quote(DIAGNOSTIC_RUN_ID)} "
    "--confirm 'I CONFIRM REAL HARDWARE CONTROL'"
)
print('\n5a) Monitor a simulation in a second PowerShell window:')
print(f"{python} monitor-run {ps_quote(database_from_config(SIMULATION_CONFIG))} --latest-running")
print('\n5b) Monitor real hardware in a second PowerShell window:')
hardware_database = database_from_config(HARDWARE_CONFIG)
if hardware_database is None:
    print('Create and verify config/hardware.local.toml first; no hardware monitor command generated.')
else:
    print(f"{python} monitor-run {ps_quote(hardware_database)} --latest-running")

## 2. 集中设置分析参数

先只修改这个 cell。默认路径为空，因此整个 Notebook 可以安全地 Run All 而不读取或写入实验数据。

- SQLite：`SOURCE_KIND='sqlite'`，填写数据库路径和 `RUN_ID`。只填写数据库、不填写 `RUN_ID` 时会列出最近 runs。
- ETO：`SOURCE_KIND='eto'`，填写单个 `.dat` 或目录，并确认 Ch1/Ch2 角色。
- 栅极标定默认关闭；只有独立标定后才填写。

In [ ]:
SOURCE_KIND = 'sqlite'                 # 'sqlite' or 'eto'
SOURCE_PATH_TEXT = ''                  # 例如 r'C:\\PPMS_Data\\ppms_control.sqlite'
RUN_ID = ''                            # SQLite 必填；留空时列出最近 runs
CHANNEL_ROLES = {1: 'xy', 2: 'xx'}     # 只用于 ETO，必须按实际接线确认
OUTPUT_DIR_TEXT = 'run_data/analysis_output'
GATE_CALIBRATION_PATH_TEXT = ''        # 默认禁用，勿直接使用 example 数值
FORMATS = ('png',)                     # 最终导出可改为 ('png', 'pdf')
RECENT_RUN_LIMIT = 20

def resolve_user_path(text: str) -> Path:
    path = Path(text).expanduser()
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()

def parameter_fingerprint() -> tuple[object, ...]:
    return (
        SOURCE_KIND, SOURCE_PATH_TEXT, RUN_ID, tuple(sorted(CHANNEL_ROLES.items())),
        OUTPUT_DIR_TEXT, GATE_CALIBRATION_PATH_TEXT, tuple(FORMATS),
    )

## 3. 列出 run 并载入数据

In [ ]:
dataset = None
calibration = None
loaded_fingerprint = None
manifest = None

if not SOURCE_PATH_TEXT.strip():
    print('尚未设置 SOURCE_PATH_TEXT。请填写上面的集中参数 cell，然后 Restart and Run All。')
else:
    source_path = resolve_user_path(SOURCE_PATH_TEXT)
    if not source_path.exists():
        raise FileNotFoundError(source_path)
    if SOURCE_KIND == 'sqlite':
        if not RUN_ID.strip():
            recent_runs = list_sqlite_runs(source_path, limit=RECENT_RUN_LIMIT)
            display(Markdown('### 最近的 SQLite runs（复制一个 run_id 到参数 cell）'))
            display(recent_runs)
        else:
            dataset = load_sqlite_run(source_path, RUN_ID.strip())
    elif SOURCE_KIND == 'eto':
        if set(CHANNEL_ROLES) != {1, 2} or not set(CHANNEL_ROLES.values()) <= {'xx', 'xy'}:
            raise ValueError('ETO CHANNEL_ROLES 必须明确把 Ch1/Ch2 映射到合法的 xx 或 xy。')
        dataset = load_eto_path(source_path, CHANNEL_ROLES)
    else:
        raise ValueError("SOURCE_KIND must be 'sqlite' or 'eto'.")

    if dataset is not None:
        if GATE_CALIBRATION_PATH_TEXT.strip():
            calibration = load_gate_calibration(resolve_user_path(GATE_CALIBRATION_PATH_TEXT))
        loaded_fingerprint = parameter_fingerprint()
        print('Loaded records:', len(dataset.records))
        print('Leakage records:', len(dataset.leakage))
        display(dict(dataset.metadata))

## 4. 先检查数据范围和质量

这里不会删除有警告的数据。出现 compliance、overload 或其他质量标记时，应先回到原始记录确认，再决定是否用于拟合或论文结论。

In [ ]:
def finite_range(values):
    finite = [float(value) for value in values if value is not None and math.isfinite(float(value))]
    return (min(finite), max(finite)) if finite else None

if dataset is None:
    print('没有载入数据；跳过质量摘要。')
else:
    flag_counts = Counter(flag for record in dataset.records for flag in record.quality_flags)
    compliance_records = [
        record for record in dataset.records
        if 'compliance' in record.comment.lower()
        or any('compliance' in flag.lower() for flag in record.quality_flags)
    ]
    summary = {
        'records': len(dataset.records),
        'signals': sorted({record.signal for record in dataset.records}),
        'harmonics': sorted({record.harmonic for record in dataset.records}),
        'temperature_K': finite_range(record.temperature_k for record in dataset.records),
        'field_T': finite_range(record.field_t for record in dataset.records),
        'drive_current_A': finite_range(record.drive_current_a for record in dataset.records),
        'frequency_Hz': finite_range(record.frequency_hz for record in dataset.records),
        'top_gate_V': finite_range(record.gate_top_voltage_v for record in dataset.records),
        'bottom_gate_V': finite_range(record.gate_bottom_voltage_v for record in dataset.records),
        'angle_deg': finite_range(record.sample_position_deg for record in dataset.records),
        'quality_flags': dict(flag_counts),
        'compliance_related_records': len(compliance_records),
    }
    display(summary)
    if compliance_records:
        display(Markdown('**警告：输入包含 compliance 相关记录；标准图会保留并标红。**'))

## 5. 生成标准图和 manifest

这一 cell 调用与 CLI `plot-data` 相同的标准实现。若修改过参数但没有重新载入，会拒绝继续，防止 Notebook 使用旧数据。

In [ ]:
if dataset is None:
    print('没有载入数据；跳过标准图生成。')
elif loaded_fingerprint != parameter_fingerprint():
    raise RuntimeError('分析参数已改变。请 Restart Kernel and Run All Cells。')
else:
    output_dir = resolve_user_path(OUTPUT_DIR_TEXT)
    manifest = generate_publication_plots(
        dataset,
        output_dir,
        calibration=calibration,
        formats=FORMATS,
    )
    print('Manifest:', manifest['manifest'])
    print('Output directory:', output_dir)

In [ ]:
if manifest is None:
    print('没有 manifest。')
else:
    rows = []
    for entry in manifest['figures']:
        rows.append({
            'key': entry['key'],
            'status': entry['status'],
            'reason': entry.get('reason', ''),
            'files': entry.get('files', []),
        })
    display(rows)
    if manifest.get('quality_warning'):
        display(Markdown(f"**Quality warning:** {manifest['quality_warning']}"))

## 6. 在 Notebook 中预览生成的 PNG

最终是否生成某张图以本次 manifest 为准，不要根据输出目录里的旧文件判断。

In [ ]:
if manifest is None:
    print('没有可预览的图。')
else:
    for entry in manifest['figures']:
        png_files = [Path(path) for path in entry.get('files', []) if str(path).lower().endswith('.png')]
        if entry['status'] == 'generated' and png_files:
            display(Markdown(f"### {entry['key']}"))
            display(Image(filename=str(png_files[0])))
    display(Markdown('### 输出文件'))
    display(FileLink(manifest['manifest']))
    display(FileLink(manifest['analysis_records_csv']))
    if manifest.get('fit_summary_csv'):
        display(FileLink(manifest['fit_summary_csv']))

## 7. 自定义探索图（非标准输出）

这里可以修改信号、谐波和横轴，快速检查数据。它不复制 `gamma`、`n-D` 或论文拟合公式；正式结果仍由上一节标准实现生成。

In [ ]:
CUSTOM_SIGNAL = 'xx'
CUSTOM_HARMONIC = 1
CUSTOM_X_FIELD = 'field_t'       # 也可用 temperature_k / drive_current_a / frequency_hz
CUSTOM_SOURCE_CONTAINS = ''

if dataset is None:
    print('没有载入数据；跳过自定义图。')
else:
    points = []
    for record in dataset.records:
        if record.signal != CUSTOM_SIGNAL or record.harmonic != CUSTOM_HARMONIC:
            continue
        if CUSTOM_SOURCE_CONTAINS and CUSTOM_SOURCE_CONTAINS not in record.source:
            continue
        x_value = getattr(record, CUSTOM_X_FIELD)
        y_value = record.voltage_v
        if x_value is not None and y_value is not None:
            points.append((float(x_value), float(y_value), record.source))
    if len(points) < 2:
        print('当前筛选不足两个点。')
    else:
        points.sort(key=lambda point: point[0])
        plt.figure(figsize=(7, 4.5))
        plt.scatter([p[0] for p in points], [p[1] * 1e6 for p in points], s=18)
        plt.xlabel(CUSTOM_X_FIELD)
        plt.ylabel(f'{CUSTOM_SIGNAL} h{CUSTOM_HARMONIC} voltage (uV)')
        plt.title('Exploratory plot — not a standardized publication panel')
        plt.tight_layout()
        plt.show()

## 8. 完成检查

- 先保存 `analysis_manifest.json`，它记录输入、通道映射、标定、生成图和跳过原因。
- ETO `2ω/3ω` 仍是无符号幅值，不能从 Notebook 恢复相位或正负号。
- 栅漏电是 Keithley 电流，不是样品输运电流。
- Notebook 显示的是离线数据；测量中的状态使用独立 `monitor-run`。
- 详细命令见 `docs/OPERATING_WORKFLOW.md`，图形定义见 `docs/DATA_ANALYSIS.md`。